# Phase 4 — Agent A4 : Discourse Analyser
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A4 (`agents/a4_discourse.py`) :
- Évaluation sur 3 dimensions : Informativité, Argumentation, Constructivité
- Production du `Score_Discours` [0-100] (moyenne des 3 dimensions)
- Identification des `high_quality_indices` (score ≥ 0.7) pour A7

> Les appels LLM sont **mockés** dans ce notebook — aucun GPU requis.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a4_discourse import a4_discourse

## 1. Corpus de test

In [ ]:
CLEANED_COMMENTS = [
    {"text": "The approach mirrors what Goodfellow describes in Deep Learning ch.9 — batch norm absent here explains training instability at 12:34.", "cleaned_text": "the approach mirrors what goodfellow describes in deep learning ch.9 — batch norm absent here explains training instability at 12:34."},
    {"text": "Great video!", "cleaned_text": "great video!"},
    {"text": "lol first", "cleaned_text": "lol first"},
    {"text": "If X implies Y then Z cannot hold simultaneously — your proof at 18:45 contradicts 5:20.", "cleaned_text": "if x implies y then z cannot hold simultaneously — your proof at 18:45 contradicts 5:20."},
    {"text": "Check my channel for more content!", "cleaned_text": "check my channel for more content!"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires (indices 0–{len(CLEANED_COMMENTS)-1})")

## 2. A4 avec LLM mocké — discours de qualité

In [ ]:
mock_response = MagicMock()
mock_response.content = json.dumps({
    "informativeness": 0.78,
    "argumentation": 0.65,
    "constructiveness": 0.80,
    "high_quality_indices": [0, 3],   # indices 0 et 3 → commentaires détaillés
    "rationale": "Two comments add substantive evidence; others are superficial.",
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

with patch("agents.a4_discourse.get_llm", return_value=mock_llm):
    result = a4_discourse(STATE)

d = result["discourse"]
avg = (d["informativeness"] + d["argumentation"] + d["constructiveness"]) / 3
print(f"Informativité    : {d['informativeness']:.1f} / 100")
print(f"Argumentation    : {d['argumentation']:.1f} / 100")
print(f"Constructivité   : {d['constructiveness']:.1f} / 100")
print(f"Score_Discours   : {d['discourse_score']:.2f} / 100  (calc={avg:.2f})")
print(f"High-quality idx : {d['high_quality_indices']}")
print()
print("Commentaires haute qualité sélectionnés pour A7 :")
for i in d["high_quality_indices"]:
    if i < len(CLEANED_COMMENTS):
        print(f"  [{i}] {CLEANED_COMMENTS[i]['text']!r}")

## 3. Fallback — LLM indisponible

In [ ]:
with patch("agents.a4_discourse.get_llm", return_value=None):
    fallback = a4_discourse(STATE)

d = fallback["discourse"]
print(f"Fallback discourse_score    : {d['discourse_score']}  (attendu: 50.0)")
print(f"Fallback high_quality_indices: {d['high_quality_indices']}  (attendu: [])")
assert d["discourse_score"] == 50.0
assert d["high_quality_indices"] == []
print("Fallback conforme.")

## Résumé A4

| Comportement | Résultat |
|---|---|
| 3 dimensions [0,1] → [0,100] | Confirmé |
| discourse_score = moyenne(D1, D2, D3) | Confirmé |
| high_quality_indices (score ≥ 0.7) | Passés à A7 pour le topic matching |
| LLM indisponible | Fallback 50.0 / indices vides |

> **Poids w2 = 0.40** — dimension la plus influente dans Score_Global (H3)